In [3]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from sklearn.cluster import KMeans
import spacy
import numpy as np

# 1) Veri & Stopword
df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8-sig")
texts = df["processed_final_review"].dropna().astype(str).tolist()

nltk.download("stopwords")
nltk.download("punkt")
# Başlangıç stopword + domain dışı filler/sentiment
extra = {
  "hotel","room","stay","place","good","nice","like","one","would","could",
  "thank","great","wonderful","problem","people","time","day","go","come",
  "take","think","say","give","look","super"
}
stop = set(stopwords.words("english")) | extra

# 2) Tokenize & Temizle
def clean_tok(doc):
    return [w for w in word_tokenize(doc.lower())
            if w.isalpha() and w not in stop]
tok_texts = [clean_tok(t) for t in texts]

# 3) LDA ile 9 topic
dictionary = Dictionary(tok_texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(t) for t in tok_texts]
lda = LdaModel(corpus, id2word=dictionary, num_topics=9, passes=15, random_state=42)

# 4) Topic başına top‐10 kelimeyi topla
keywords = set()
for t in range(9):
    kws = [w for w,_ in lda.show_topic(t, topn=10)]
    keywords.update(kws)
keywords = list(keywords)

print(f"LDA’dan gelen keyword sayısı: {len(keywords)}")
print(keywords)

# 5) spaCy embedding + KMeans (7 cluster)
nlp = spacy.load("en_core_web_lg")
embs, valid = [], []
for w in keywords:
    tok = nlp(w)
    if tok.has_vector:
        embs.append(tok.vector)
        valid.append(w)
X = np.vstack(embs)

km = KMeans(n_clusters=7, random_state=42).fit(X)
clusters = {i: [] for i in range(7)}
for w, lbl in zip(valid, km.labels_):
    clusters[lbl].append(w)

# 6) Son hali Print et
print("\n🏷️ Son Otelcilik‑Aspect Kümeleriniz:")
for cid, kws in clusters.items():
    print(f"  Aspect {cid+1}:", kws)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


LDA’dan gelen keyword sayısı: 56
['pool', 'staff', 'balcony', 'breakfast', 'everything', 'next', 'shower', 'beach', 'water', 'air', 'condition', 'house', 'restaurant', 'help', 'sea', 'phone', 'accommodation', 'person', 'dirty', 'family', 'holiday', 'apart', 'money', 'clean', 'friend', 'door', 'pay', 'akyaka', 'apartment', 'small', 'night', 'kitchen', 'mind', 'owner', 'need', 'river', 'call', 'make', 'leave', 'price', 'floor', 'work', 'peace', 'beautiful', 'check', 'stop', 'see', 'friendly', 'comfortable', 'want', 'walk', 'location', 'towel', 'dog', 'recommend', 'bed']

🏷️ Son Otelcilik‑Aspect Kümeleriniz:
  Aspect 1: ['holiday']
  Aspect 2: ['staff', 'air', 'condition', 'help', 'money', 'clean', 'door', 'small', 'need', 'work', 'peace', 'check', 'towel', 'recommend']
  Aspect 3: ['balcony', 'shower', 'apartment', 'kitchen', 'floor', 'bed']
  Aspect 4: ['everything', 'next', 'house', 'phone', 'person', 'dirty', 'family', 'apart', 'friend', 'pay', 'night', 'mind', 'owner', 'call', 'make'

In [4]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
import spacy
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 1) Veri & stopword’ler
nltk.download("stopwords")
nltk.download("punkt")
df = pd.read_csv(
    "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv",
    encoding="utf-8-sig"
)
texts = df["processed_final_review"].dropna().astype(str).tolist()

extra = {
    "hotel","room","stay","place","good","nice","like","one","would","could",
    "thank","great","wonderful","problem","people","time","day","go","come",
    "take","think","say","give","look","super"
}
stop = set(stopwords.words("english")) | extra

# 2) Tokenize & temizle
def clean_tok(doc):
    return [
        w for w in word_tokenize(doc.lower())
        if w.isalpha() and w not in stop
    ]
tok_texts = [clean_tok(t) for t in texts]

# 3) En iyi topic sayısını coherence ile ara
coherences = []
for k in range(2, 11):
    dct = Dictionary(tok_texts)
    dct.filter_extremes(no_below=5, no_above=0.5)
    corpus = [dct.doc2bow(t) for t in tok_texts]
    lda = LdaModel(corpus, id2word=dct, num_topics=k, passes=15, random_state=42)
    cm = CoherenceModel(
        model=lda, texts=tok_texts, dictionary=dct, coherence='c_v'
    )
    coherences.append((k, cm.get_coherence()))

# coherence sonuçlarını bastır
print("Optimal topic sayısı arama (k, coherence):")
for k, c in coherences:
    print(f"  {k:2d} → {c:.4f}")

best_k = max(coherences, key=lambda x: x[1])[0]
print(f"\nEn iyi topic sayısı = {best_k}\n")

# 4) En iyi k ile LDA modelini eğit
dictionary = Dictionary(tok_texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(t) for t in tok_texts]
lda = LdaModel(corpus, id2word=dictionary, num_topics=best_k, passes=15, random_state=42)

# 5) Topic başına top-10 kelimeyi topla
keywords = set()
print("==== LDA’dan çıkan topic kelimeleri ====")
for t in range(best_k):
    terms = [w for w,_ in lda.show_topic(t, topn=10)]
    print(f"Topic {t+1:2d}:", terms)
    keywords.update(terms)
keywords = list(keywords)

# 6) spaCy embeddings + KMeans
nlp = spacy.load("en_core_web_lg")
embs, valid = [], []
for w in keywords:
    tok = nlp(w)
    if tok.has_vector:
        embs.append(tok.vector)
        valid.append(w)
X = np.vstack(embs)

# burada kaç aspect görmek istiyorsanız onu seçin; ör. 5
n_clusters = 5
km = KMeans(n_clusters=n_clusters, random_state=42).fit(X)
clusters = {i: [] for i in range(n_clusters)}
for w, lbl in zip(valid, km.labels_):
    clusters[lbl].append(w)

# 7) Sonuçları göster
print(f"\n🏷️ Otelcilik‑Aspect Kümeleriniz (k={n_clusters}):")
for cid, kws in clusters.items():
    print(f"  Aspect {cid+1}:", kws)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Optimal topic sayısı arama (k, coherence):
   2 → 0.4318
   3 → 0.3657
   4 → 0.3705
   5 → 0.3650
   6 → 0.3894
   7 → 0.3917
   8 → 0.3891
   9 → 0.3747
  10 → 0.3593

En iyi topic sayısı = 2

==== LDA’dan çıkan topic kelimeleri ====
Topic  1: ['clean', 'akyaka', 'breakfast', 'location', 'friendly', 'recommend', 'family', 'beautiful', 'staff', 'beach']
Topic  2: ['breakfast', 'clean', 'night', 'price', 'location', 'bed', 'bathroom', 'shower', 'water', 'leave']

🏷️ Otelcilik‑Aspect Kümeleriniz (k=5):
  Aspect 1: ['breakfast', 'bed']
  Aspect 2: ['beach', 'beautiful']
  Aspect 3: ['staff', 'friendly']
  Aspect 4: ['night', 'location', 'family', 'water', 'leave', 'price', 'clean', 'recommend']
  Aspect 5: ['bathroom', 'shower']


In [5]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
import spacy
import numpy as np
from sklearn.cluster import KMeans

# 1) Veri & stopword
nltk.download("punkt")
nltk.download("stopwords")
df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8-sig")
texts = df["processed_final_review"].dropna().astype(str).tolist()

extra = {"hotel","room","stay","place","good","nice","like","one","would","could"}
stop = set(stopwords.words("english")) | extra

# 2) Tokenize & temizle
def clean_tok(doc):
    return [w for w in word_tokenize(doc.lower())
            if w.isalpha() and w not in stop]
tok_texts = [clean_tok(t) for t in texts]

# 3) En iyi k yerine doğrudan k listesini deneyelim
dictionary = Dictionary(tok_texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(t) for t in tok_texts]

# coherence sonuçlarına bakmak istersen:
# for k in [3,4,5,6]:
#     lda = LdaModel(corpus, id2word=dictionary, num_topics=k, passes=15, random_state=42)
#     cm = CoherenceModel(model=lda, texts=tok_texts, dictionary=dictionary, coherence='c_v')
#     print(k, cm.get_coherence())

# 4) Burada örnek: k=6, 6 topic
lda = LdaModel(corpus, id2word=dictionary, num_topics=6, passes=15, random_state=42)

# 5) Topic’lerden keyword toplama
keywords = set()
for t in range(6):
    kws = [w for w,_ in lda.show_topic(t, topn=10)]
    print(f"Topic {t+1}:", kws)
    keywords.update(kws)
keywords = list(keywords)

# 6) spaCy vector + KMeans
nlp = spacy.load("en_core_web_lg")
embs, valid = [], []
for w in keywords:
    tok = nlp(w)
    if tok.has_vector:
        embs.append(tok.vector)
        valid.append(w)
X = np.vstack(embs)

n_clusters = 5   # filtre sayınıza göre 4-6 arası seçin
km = KMeans(n_clusters=n_clusters, random_state=42).fit(X)

# 7) Kümeleri yazdır
clusters = {i: [] for i in range(n_clusters)}
for w,lbl in zip(valid, km.labels_):
    clusters[lbl].append(w)

print("\nOtelcilik‑Aspect Kümeleri:")
for i,kw in clusters.items():
    print(f" Aspect {i+1}:", kw)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Topic 1: ['clean', 'akyaka', 'thank', 'location', 'friendly', 'beautiful', 'staff', 'walk', 'recommend', 'breakfast']
Topic 2: ['breakfast', 'beach', 'clean', 'great', 'akyaka', 'location', 'apartment', 'staff', 'restaurant', 'well']
Topic 3: ['clean', 'recommend', 'family', 'people', 'pool', 'thank', 'go', 'akyaka', 'business', 'apart']
Topic 4: ['say', 'night', 'breakfast', 'clean', 'price', 'sleep', 'go', 'bed', 'location', 'come']
Topic 5: ['breakfast', 'bathroom', 'shower', 'night', 'say', 'think', 'clean', 'give', 'work', 'water']
Topic 6: ['thank', 'go', 'family', 'house', 'sea', 'want', 'holiday', 'take', 'apartment', 'clean']

Otelcilik‑Aspect Kümeleri:
 Aspect 1: ['pool', 'beach', 'water', 'sea']
 Aspect 2: ['restaurant', 'recommend']
 Aspect 3: ['staff', 'house', 'family', 'holiday', 'night', 'walk']
 Aspect 4: ['come', 'thank', 'say', 'go', 'people', 'apart', 'clean', 'business', 'take', 'think', 'price', 'work', 'beautiful', 'well', 'great', 'friendly', 'want', 'give', 'lo

In [7]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
import spacy
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1) Veri & Stopword
nltk.download("punkt")
nltk.download("stopwords")
df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8-sig")
texts = df["processed_final_review"].dropna().astype(str).tolist()

base_stop = set(stopwords.words("english"))
extra = {"hotel","room","stay","place","good","nice","like","one","would","could",
         "come","say","go","thank","want","take","think","great","well"}
stop = base_stop | extra

# 2) Tokenize & temizle
def clean_tok(doc):
    return [w for w in word_tokenize(doc.lower())
            if w.isalpha() and w not in stop]
tok_texts = [clean_tok(t) for t in texts]

# 3) En iyi topic sayısını coherence ile seç (opsiyonel)
for k in [4,5,6,7,8]:
     lda_tmp = LdaModel([dictionary.doc2bow(t) for t in tok_texts],
                        id2word=dictionary, num_topics=k, passes=15, random_state=42)
     cv = CoherenceModel(model=lda_tmp, texts=tok_texts,
                         dictionary=dictionary, coherence='c_v').get_coherence()
     print(k, cv)

# 4) LDA (örnek: 6 topic)
dictionary = Dictionary(tok_texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(t) for t in tok_texts]
lda = LdaModel(corpus, id2word=dictionary, num_topics=6,
               passes=15, random_state=42)

# 5) Topic keyword’lerini topla
keywords = set()
for t in range(6):
    kws = [w for w,_ in lda.show_topic(t, topn=10)]
    print(f"Topic {t+1}:", kws)
    keywords.update(kws)
keywords = list(keywords)

# 6) spaCy vector + Silhouette optimizasyonlu KMeans
nlp = spacy.load("en_core_web_lg")
embs, valid = [], []
for w in keywords:
    tok = nlp(w)
    if tok.has_vector:
        embs.append(tok.vector)
        valid.append(w)
X = np.vstack(embs)

# 7) En iyi n_clusters’ı bul
scores = []
for k in range(3,8):
    km_tmp = KMeans(n_clusters=k, random_state=42).fit(X)
    scores.append((k, silhouette_score(X, km_tmp.labels_)))
best_k = max(scores, key=lambda x: x[1])[0]
print("Best n_clusters by silhouette:", best_k)

# 8) Son olarak en iyi k ile kümele ve yazdır
km = KMeans(n_clusters=best_k, random_state=42).fit(X)
clusters = {i: [] for i in range(best_k)}
for w,lbl in zip(valid, km.labels_):
    clusters[lbl].append(w)

print("\n🎯 Son Otelcilik‑Aspect Kümeleri:")
for i,kw in clusters.items():
    print(f" Aspect {i+1}:", kw)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


4 0.3686939971114094
5 0.3942336752539502
6 0.3823450822784687
7 0.38523157433891786
8 0.3579318751383709
Topic 1: ['clean', 'location', 'apartment', 'staff', 'friendly', 'day', 'akyaka', 'price', 'family', 'problem']
Topic 2: ['night', 'bed', 'shower', 'water', 'clean', 'give', 'pool', 'leave', 'reception', 'day']
Topic 3: ['akyaka', 'walk', 'apartment', 'location', 'clean', 'distance', 'azmak', 'beach', 'river', 'child']
Topic 4: ['clean', 'smell', 'money', 'night', 'dirty', 'breakfast', 'recommend', 'give', 'location', 'call']
Topic 5: ['breakfast', 'clean', 'akyaka', 'friendly', 'beautiful', 'staff', 'location', 'family', 'recommend', 'make']
Topic 6: ['apart', 'clean', 'bey', 'holiday', 'breakfast', 'interest', 'akyaka', 'view', 'sea', 'family']
Best n_clusters by silhouette: 3

🎯 Son Otelcilik‑Aspect Kümeleri:
 Aspect 1: ['staff', 'breakfast', 'smell', 'beach', 'child', 'distance', 'dirty', 'family', 'holiday', 'apart', 'view', 'money', 'clean', 'apartment', 'night', 'interest', 

In [20]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from gensim.models.phrases import Phrases, Phraser
import spacy
import numpy as np
from sklearn.cluster import KMeans

# 1) İndirilecek NLTK paketleri
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

# 2) veriyi yükle
df = pd.read_csv(
    "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv",
    encoding="utf-8-sig"
)
docs = df["processed_final_review"].dropna().astype(str).tolist()

# 3) ek stop‐word’ler
extra = {"hotel","room","stay","place","good","nice","like","one","would","could"}
stop = set(stopwords.words("english")) | extra

# 4) spaCy modelini yükle
nlp = spacy.load("en_core_web_lg", disable=["parser","ner"])

# 5) Tokenizasyon + POS‑filtre + lemmatizasyon
def spacy_tokenize(text):
    doc = nlp(text.lower())
    tokens = []
    for t in doc:
        # sadece somut isimler/proper isimler ve sıfatlar, alfa‐tokenler,
        # stop‐word listesinde olmayanlar
        if (
            t.is_alpha and
            t.lemma_ not in stop and
            t.pos_ in ("NOUN","PROPN","ADJ")
        ):
            tokens.append(t.lemma_)
    return tokens

tok_texts = [spacy_tokenize(d) for d in docs]

# 6) bigram/fazla kelime gruplarını yakala (ör: “river view”, “breakfast buffet”)
phrases = Phrases(tok_texts, min_count=10, threshold=5)
bigram = Phraser(phrases)
tok_texts = [bigram[t] for t in tok_texts]

# 7) sözlük ve corpus
dictionary = Dictionary(tok_texts)
dictionary.filter_extremes(no_below=10, no_above=0.5)
corpus = [dictionary.doc2bow(t) for t in tok_texts]

# 8) optimal topic sayısını coherence ile bul (3–8 arası)
best_k, best_coh = None, -1
for k in range(3,9):
    model = LdaModel(corpus, id2word=dictionary, num_topics=k,
                     passes=15, random_state=42)
    coh = CoherenceModel(model=model, texts=tok_texts,
                         dictionary=dictionary, coherence="c_v")\
         .get_coherence()
    print(f" k={k:2d} → coherence={coh:.4f}")
    if coh > best_coh:
        best_k, best_coh = k, coh

print(f"\n👉 En iyi topic sayısı = {best_k}, coherence={best_coh:.4f}")

# 9) final LDA
lda = LdaModel(corpus, id2word=dictionary,
               num_topics=best_k, passes=15, random_state=42)

# 10) Topic’lerden top‐10 kelimeyi al
keywords = set()
for t in range(best_k):
    kw = [w for w,_ in lda.show_topic(t, topn=10)]
    print(f"Topic {t+1}:", kw)
    keywords.update(kw)
keywords = list(keywords)

# 11) spaCy embedding + KMeans
embs, valid = [], []
for w in keywords:
    tok = nlp(w)
    if tok.has_vector:
        embs.append(tok.vector)
        valid.append(w)
X = np.stack(embs)

n_clusters = 5
km = KMeans(n_clusters=n_clusters, random_state=42).fit(X)

# 12) sonuçları yazdır
clusters = {i: [] for i in range(n_clusters)}
for w,l in zip(valid, km.labels_):
    clusters[l].append(w)

print("\n🔑 Otelcilik‑Aspect Kümeleri:")
for i,kw in clusters.items():
    print(f" Aspect {i+1}:", kw)


 k= 3 → coherence=0.4066
 k= 4 → coherence=0.3777
 k= 5 → coherence=0.4071
 k= 6 → coherence=0.4083
 k= 7 → coherence=0.3965
 k= 8 → coherence=0.4031

👉 En iyi topic sayısı = 6, coherence=0.4083
Topic 1: ['breakfast', 'clean', 'day', 'night', 'location', 'reception', 'price', 'bed', 'water', 'service']
Topic 2: ['great', 'beautiful', 'clean', 'holiday', 'breakfast', 'beach', 'akyaka', 'comfortable', 'friendly', 'family']
Topic 3: ['clean', 'location', 'akyaka', 'sea', 'apartment', 'walk_distance', 'small', 'price', 'breakfast', 'spacious']
Topic 4: ['clean', 'location', 'price', 'shower', 'floor', 'beautiful', 'people', 'bathroom', 'door', 'pool']
Topic 5: ['night', 'apartment', 'breakfast', 'air_condition', 'bed', 'bad', 'toilet', 'sleep', 'small', 'internet']
Topic 6: ['akyaka', 'clean', 'family', 'pool', 'owner', 'thank', 'location', 'child', 'holiday', 'day']

🔑 Otelcilik‑Aspect Kümeleri:
 Aspect 1: ['child', 'family', 'holiday', 'owner']
 Aspect 2: ['breakfast', 'sleep', 'night', 

In [21]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy
from collections import Counter
import numpy as np

# 1) Veri & Ön hazırlık
nltk.download("punkt")
nltk.download("stopwords")

df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8-sig")
texts = df["processed_final_review"].dropna().astype(str).tolist()

# 2) Frekans bazlı token havuzu (en sık 1000 kelime)
stop = set(stopwords.words("english"))
all_tokens = []
for t in texts:
    all_tokens += [w.lower() for w in word_tokenize(t) if w.isalpha() and w.lower() not in stop]
freq = Counter(all_tokens)
vocab = [w for w,_ in freq.most_common(1000)]

# 3) spaCy embedding modeli
nlp = spacy.load("en_core_web_lg")

# 4) Seed aspect’ler
aspects = ["room", "service", "location", "price", "food"]
aspect_vecs = {asp: nlp(asp).vector for asp in aspects}

# 5) Her kelimenin vektörünü hazırla
word_vecs = {}
for w in vocab:
    tok = nlp(w)
    if tok.has_vector:
        word_vecs[w] = tok.vector

# 6) Benzerlikleri hesapla ve eşik üstündekileri topla
threshold = 0.60
aspect_lexicons = {}
for asp, a_vec in aspect_vecs.items():
    sims = []
    for w, v in word_vecs.items():
        sim = np.dot(a_vec, v) / (np.linalg.norm(a_vec)*np.linalg.norm(v))
        if sim >= threshold:
            sims.append((w, sim))
    # benzerlik skoruna göre sırala, en yüksek 20’yi al
    sims = sorted(sims, key=lambda x: x[1], reverse=True)[:20]
    aspect_lexicons[asp] = [w for w,_ in sims]

# 7) Sonuç
for asp, kws in aspect_lexicons.items():
    print(f"{asp:10} → {kws}")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


room       → ['room', 'downstairs', 'floor', 'bathroom', 'bedroom', 'desk', 'kitchen', 'balcony', 'bed', 'spacious', 'house', 'laundry', 'apartment', 'sofa']
service    → ['service', 'customer']
location   → ['location', 'proximity', 'area']
price      → ['price', 'cost', 'buy', 'discount']
food       → ['food', 'meal', 'eat', 'restaurant', 'delicious', 'soup']


In [23]:
import pandas as pd
import nltk, spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from sklearn.cluster import KMeans
import numpy as np

# 1) Gerekli verileri indir
nltk.download("punkt")
nltk.download("stopwords")

# 2) Veriyi oku ve temizlenmiş yorumları al
df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8-sig")
texts = df["processed_final_review"].dropna().astype(str).tolist()

# 3) Stop‑word set’i (NLTK + domain dışı filler)
extra_stop = {
    "hotel","room","stay","place","good","nice","like","one","would","could",
    "thank","great","wonderful","problem","people","time","day","go","come",
    "take","think","say","give","look","super"
}
stop = set(stopwords.words("english")) | extra_stop

# 4) spaCy modelini yükle
nlp = spacy.load("en_core_web_lg")

# ——————————————————————————————————————————
# A) Seed‑Lexicon Oluşturma (semantic benzerliğe göre)
# ——————————————————————————————————————————

# 4.a) Tüm yorumlardan en sık 1000 kelimeyi çıkar
all_tokens = []
for text in texts:
    for w in word_tokenize(text.lower()):
        if w.isalpha() and w not in stop and nlp(w).has_vector:
            all_tokens.append(w)
freq = pd.Series(all_tokens).value_counts().head(1000).index.tolist()

# 4.b) Seed aspect’leri & vektörleri
aspect_seeds = ["room","service","location","price","food"]
asp_vecs = {a: nlp(a).vector for a in aspect_seeds}

# 4.c) Her seed’e en çok benzeyen top‑15 kelimeyi al (sim ≥ 0.6 eşiğiyle)
seed_lexicons = {}
for a, vec in asp_vecs.items():
    sims = []
    for w in freq:
        w_vec = nlp(w).vector
        score = vec.dot(w_vec)/(np.linalg.norm(vec)*np.linalg.norm(w_vec))
        if score >= 0.6:
            sims.append((w, score))
    # en yüksekten aza sırala, sadece isim (NOUN) ve proper isim (PROPN) bırak
    filtered = [
        w for w,_ in sorted(sims, key=lambda x: -x[1])
        if nlp(w)[0].pos_ in {"NOUN","PROPN"}
    ][:15]
    seed_lexicons[a] = filtered

# ——————————————————————————————————————————
# B) LDA + KMeans ile Ek Aspect Önerisi
# ——————————————————————————————————————————

# 4.d) Tokenize & temizle (LDA’ya özel)
tok_texts = [
    [w for w in word_tokenize(doc.lower()) if w.isalpha() and w not in stop]
    for doc in texts
]

# 4.e) Dictionary & Corpus
dct = Dictionary(tok_texts)
dct.filter_extremes(no_below=5, no_above=0.5)
corp = [dct.doc2bow(doc) for doc in tok_texts]

# 4.f) Optimal topic sayısını coherence ile bul (3–8 arası deneyelim)
coh = {}
for k in range(3,9):
    model = LdaModel(corp, id2word=dct, num_topics=k, passes=10, random_state=42)
    cm = CoherenceModel(model=model, texts=tok_texts, dictionary=dct, coherence='c_v')
    coh[k] = cm.get_coherence()
best_k = max(coh, key=coh.get)
print(f"Optimal topic sayısı = {best_k}, coherence = {coh[best_k]:.4f}")

# 4.g) O topic sayısıyla LDA modelini eğit
lda = LdaModel(corp, id2word=dct, num_topics=best_k, passes=15, random_state=42)

# 4.h) Topic’lerden top‑10 kelime çek
lda_keywords = set()
for t in range(best_k):
    toks = [w for w,_ in lda.show_topic(t, topn=10)]
    lda_keywords.update(toks)
lda_keywords = list(lda_keywords)

# 4.i) Yine yalnızca isim ve proper isim bırak
clean_kw = [w for w in lda_keywords
            if nlp(w)[0].pos_ in {"NOUN","PROPN"} and nlp(w).has_vector]

# 4.j) Bu keyword’leri spaCy vektörleriyle KMeans’e sok
embs = np.vstack([nlp(w).vector for w in clean_kw])
n_clusters = 3  # yeni aspect sayısı
km = KMeans(n_clusters=n_clusters, random_state=42).fit(embs)

# 4.k) KMeans çıktısını al
lda_clusters = {i: [] for i in range(n_clusters)}
for w,label in zip(clean_kw, km.labels_):
    lda_clusters[label].append(w)

# ——————————————————————————————————————————
# 5) Sonuçları bastır
# ——————————————————————————————————————————

print("\n=== Seed‑Lexicons ===")
for a, words in seed_lexicons.items():
    print(f"{a:10}: {words}")

print("\n=== LDA‑Clusters (Ek Aspect’ler) ===")
for idx, words in lda_clusters.items():
    print(f"Aspect_{idx+1:1d}: {words}")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Optimal topic sayısı = 7, coherence = 0.3806

=== Seed‑Lexicons ===
room      : ['floor', 'bathroom', 'bedroom', 'desk', 'kitchen', 'balcony', 'bed', 'house', 'laundry', 'apartment', 'basement', 'sofa']
service   : ['service', 'customer']
location  : ['location', 'proximity', 'area']
price     : ['price', 'cost']
food      : ['food', 'meal', 'restaurant', 'lunch', 'soup']

=== LDA‑Clusters (Ek Aspect’ler) ===
Aspect_1: ['beach', 'water', 'sea', 'river']
Aspect_2: ['pool', 'balcony', 'breakfast', 'sleep', 'house', 'distance', 'family', 'holiday', 'money', 'apartment', 'night', 'kitchen', 'mind', 'price', 'peace', 'bed']
Aspect_3: ['staff', 'quality', 'owner', 'location', 'customer']


In [25]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
import spacy
from sklearn.cluster import KMeans

# 1) Veri yükle & stopword hazirla
nltk.download("punkt")
nltk.download("stopwords")

df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8-sig")
texts = df["processed_final_review"].dropna().astype(str).tolist()

# Temel stopword’ler + domain genel + ek gürültü kelimeleri
base_stop    = set(stopwords.words("english"))
domain_stop  = {"hotel","room","stay","place","good","nice","like","one","marmaris", "bey", "super" }
general_stop = {"clean","thank","go","say","would","could"}
manual_stop  = {"river","mind"}  # sıkça gürültüye neden olanlar
stop_words   = base_stop | domain_stop | general_stop | manual_stop

# 2) Tokenize + POS‑filtre
nlp = spacy.load("en_core_web_lg")
def preprocess(doc):
    tokens = []
    for w in word_tokenize(doc.lower()):
        if w.isalpha() and w not in stop_words:
            tok = nlp(w)[0]
            if tok.pos_ in ["NOUN","PROPN"]:
                tokens.append(w)
    return tokens

tok_texts = [preprocess(doc) for doc in texts]

# 3) Optimal topic sayısını coherence ile bul ve LDA eğit
dictionary = Dictionary(tok_texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus     = [dictionary.doc2bow(doc) for doc in tok_texts]

best_k, best_coh = None, -1
for k in [5, 6, 7]:
    lda = LdaModel(corpus, id2word=dictionary, num_topics=k, passes=15, random_state=42)
    coh = CoherenceModel(model=lda, texts=tok_texts, dictionary=dictionary, coherence="c_v").get_coherence()
    print(f"k={k} → coherence={coh:.4f}")
    if coh > best_coh:
        best_k, best_coh = k, coh

print(f"\nSeçilen topic sayısı = {best_k}, coherence={best_coh:.4f}\n")
lda = LdaModel(corpus, id2word=dictionary, num_topics=best_k, passes=15, random_state=42)

# 4) Topic’lerden anahtar kelimeleri topla, genel stopword’leri çıkar
keywords = set()
for t in range(best_k):
    kws = [w for w,_ in lda.show_topic(t, topn=15)]
    keywords.update(kws)
keywords = [w for w in keywords if w not in stop_words]

# 5) spaCy ile vektörleri al, KMeans ile kümelere ayır, her kümeden top 10’u al
embs, words = [], []
for w in keywords:
    tok = nlp(w)
    if tok.has_vector:
        embs.append(tok.vector)
        words.append(w)
embs = np.vstack(embs)

n_clusters = 5
km = KMeans(n_clusters=n_clusters, random_state=42).fit(embs)
centroids = km.cluster_centers_

clusters = {i: [] for i in range(n_clusters)}
for i, w in enumerate(words):
    lbl = km.labels_[i]
    # kelime–centroid benzerliği
    sim = float(np.dot(embs[i], centroids[lbl]) / 
                (np.linalg.norm(embs[i]) * np.linalg.norm(centroids[lbl])))
    clusters[lbl].append((w, sim))

# her kümeden en yüksek sim’li 10 kelimeyi seç
for lbl in clusters:
    topw = sorted(clusters[lbl], key=lambda x: -x[1])[:10]
    clusters[lbl] = [w for w,_ in topw]

# 6) Seed‑lexicon’ları semantic similarity ile genişlet
#    Sık geçen ilk 1000 token içinden POS filtresiyle seçip her seed’e bak
freq_tokens = pd.Series([w for doc in tok_texts for w in doc]).value_counts().index.tolist()[:1000]
seed_aspects = ["room","service","location","price","food"]
lexicons = {}
for seed in seed_aspects:
    seed_vec = nlp(seed).vector
    sims = []
    for w in freq_tokens:
        tok = nlp(w)[0]
        if tok.has_vector and tok.pos_ in ["NOUN","PROPN"]:
            s = float(np.dot(seed_vec, tok.vector) /
                      (np.linalg.norm(seed_vec)*np.linalg.norm(tok.vector)))
            if s >= 0.55:
                sims.append((w, s))
    # en iyi 12 terimi al
    lexicons[seed] = [w for w,_ in sorted(sims, key=lambda x: -x[1])[:12]]

# 7) Sonuçları yazdır
print("=== Seed‑Lexicons ===")
for asp, kws in lexicons.items():
    print(f"{asp:10}: {kws}")

print("\n=== Discover’d Aspects (Clusters) ===")
for lbl, kws in clusters.items():
    print(f"Aspect {lbl+1}: {kws}")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


k=5 → coherence=0.5115
k=6 → coherence=0.4815
k=7 → coherence=0.5046

Seçilen topic sayısı = 5, coherence=0.5115

=== Seed‑Lexicons ===
room      : ['floor', 'bathroom', 'bedroom', 'desk', 'hall', 'kitchen', 'balcony', 'bed', 'house', 'laundry', 'apartment', 'basement']
service   : ['service', 'customer', 'maintenance', 'business']
location  : ['location', 'proximity', 'area', 'destination', 'vicinity']
price     : ['price', 'cost', 'value']
food      : ['food', 'meal', 'meat', 'restaurant', 'lunch', 'diet', 'soup', 'bread', 'grocery', 'pizza', 'dinner', 'dessert']

=== Discover’d Aspects (Clusters) ===
Aspect 1: ['sea', 'beach', 'water']
Aspect 2: ['apartment', 'house', 'accommodation', 'restaurant']
Aspect 3: ['time', 'home', 'day', 'people', 'family', 'service', 'business', 'car', 'location', 'food']
Aspect 4: ['bathroom', 'shower', 'toilet', 'towel', 'pool', 'door', 'cleanliness']
Aspect 5: ['bed', 'floor', 'breakfast']


In [26]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
import spacy
from sklearn.cluster import KMeans
from keybert import KeyBERT

# 1) Veri yükle & stopwords hazırlığı
nltk.download('punkt')
nltk.download('stopwords')

df = pd.read_csv(
    "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv",
    encoding="utf-8-sig"
)
texts = df["processed_final_review"].dropna().astype(str).tolist()

# Temel + domain + genel + manuel stopword setleri
base_stop    = set(stopwords.words('english'))
domain_stop  = {"hotel","room","stay","place","good","nice","like","would","could"}
general_stop = {"clean","thank","go","say","want","take","think"}
manual_stop  = {"time","home","people","business","car","river","mind"}
stop_words   = base_stop | domain_stop | general_stop | manual_stop

# 2) Ön işleme: POS filtresi (NOUN, PROPN, ADJ)
nlp = spacy.load('en_core_web_lg')
def preprocess(doc):
    tokens = []
    for w in word_tokenize(doc.lower()):
        if w.isalpha() and w not in stop_words:
            tok = nlp(w)[0]
            if tok.pos_ in ['NOUN','PROPN','ADJ']:
                tokens.append(w)
    return tokens

tok_texts = [preprocess(doc) for doc in texts]

# 3) Optimal topic sayısını coherence ile belirle ve LDA eğit

dictionary = Dictionary(tok_texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus     = [dictionary.doc2bow(doc) for doc in tok_texts]

best_k, best_coh = None, -1.0
for k in [5,6,7]:
    lda = LdaModel(corpus, id2word=dictionary, num_topics=k, passes=15, random_state=42)
    coh = CoherenceModel(model=lda, texts=tok_texts, dictionary=dictionary, coherence='c_v')
    score = coh.get_coherence()
    print(f"k={k} → coherence={score:.4f}")
    if score > best_coh:
        best_k, best_coh = k, score
print(f"\nSeçilen topic sayısı = {best_k}, coherence={best_coh:.4f}\n")
lda = LdaModel(corpus, id2word=dictionary, num_topics=best_k, passes=15, random_state=42)

# 4) LDA’dan anahtar kelimeleri topla, stop_words çıkar
keywords = set()
for t in range(best_k):
    kws = [w for w,_ in lda.show_topic(t, topn=15)]
    keywords.update(kws)
keywords = [w for w in keywords if w not in stop_words]

# 5) Kelimeleri spaCy embedding + KMeans clustering ile gruplandır
embs, words = [], []
for w in keywords:
    tok = nlp(w)
    if tok.has_vector:
        embs.append(tok.vector)
        words.append(w)
embs = np.vstack(embs)

n_clusters = 6
km = KMeans(n_clusters=n_clusters, random_state=42).fit(embs)
centroids = km.cluster_centers_
clusters = {i: [] for i in range(n_clusters)}
for i, w in enumerate(words):
    lbl = km.labels_[i]
    sim = float(np.dot(embs[i], centroids[lbl]) /
                (np.linalg.norm(embs[i]) * np.linalg.norm(centroids[lbl])))
    clusters[lbl].append((w, sim))
# Her kümeden top10'u seç
for lbl in clusters:
    clusters[lbl] = [w for w,_ in sorted(clusters[lbl], key=lambda x: -x[1])[:10]]

# 6) Seed‑lexicon genişletme (eşik = 0.60)
freq_tokens = pd.Series(
    [w for doc in tok_texts for w in doc]
).value_counts().index.tolist()[:1000]
seed_aspects = ['room','service','location','price','food']
lexicons = {}
for seed in seed_aspects:
    seed_vec = nlp(seed).vector
    sims = []
    for w in freq_tokens:
        if w in stop_words: continue
        tok = nlp(w)[0]
        if tok.has_vector and tok.pos_ in ['NOUN','PROPN','ADJ']:
            s = float(np.dot(seed_vec, tok.vector) /
                      (np.linalg.norm(seed_vec)*np.linalg.norm(tok.vector)))
            if s >= 0.60:
                sims.append((w, s))
    lexicons[seed] = [w for w,_ in sorted(sims, key=lambda x: -x[1])[:12]]

# 7) Keyphrase önerisi (2-3 kelimelik)
kw_model = KeyBERT('all-MiniLM-L6-v2')
keyphrases = []
for doc in texts:
    noun_chunks = [chunk.text for chunk in nlp(doc).noun_chunks]
    kws = kw_model.extract_keywords(
        ' '.join(noun_chunks),
        top_n=3,
        keyphrase_ngram_range=(2,3),
        stop_words=stop_words
    )
    keyphrases += [kw for kw,_ in kws]
from collections import Counter
top_phrases = [kw for kw,_ in Counter(keyphrases).most_common(15)]

# 8) Sonuçları yazdır
print("=== Seed‑Lexicons ===")
for asp, kws in lexicons.items():
    print(f"{asp:10}: {kws}")
print("\n=== Discovered Aspects (Clusters) ===")
for lbl, kws in clusters.items():
    print(f"Aspect {lbl+1}: {kws}")
print("\n=== Suggested Keyphrases ===", top_phrases)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


k=5 → coherence=0.4571
k=6 → coherence=0.4689
k=7 → coherence=0.4638

Seçilen topic sayısı = 6, coherence=0.4689

=== Seed‑Lexicons ===
room      : ['floor', 'bathroom', 'bedroom', 'desk', 'hall', 'kitchen', 'balcony', 'bed', 'spacious', 'house', 'laundry', 'apartment']
service   : ['service', 'customer']
location  : ['location', 'proximity', 'area']
price     : ['price', 'cost']
food      : ['food', 'meal', 'meat', 'restaurant', 'delicious', 'lunch', 'tasty', 'soup']

=== Discovered Aspects (Clusters) ===
Aspect 1: ['family', 'holiday', 'owner']
Aspect 2: ['bathroom', 'kitchen', 'shower', 'floor', 'toilet', 'balcony', 'bed', 'house', 'apartment', 'garden']
Aspect 3: ['customer', 'service', 'quality', 'cleanliness']
Aspect 4: ['friendly', 'breakfast', 'restaurant', 'comfortable', 'staff']
Aspect 5: ['great', 'lot', 'day', 'wonderful', 'bad', 'next', 'night', 'side', 'small', 'beautiful']
Aspect 6: ['beach', 'pool', 'sea', 'water', 'park']

=== Suggested Keyphrases === []


In [27]:
import pandas as pd
import numpy as np
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag
from nltk.chunk import RegexpParser

from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

from sklearn.cluster import KMeans

from keybert import KeyBERT

# 1) Download & load models
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger")

nlp = spacy.load("en_core_web_lg")
kw_model = KeyBERT("all-MiniLM-L6-v2")

# 2) Load your data
df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv",
                 encoding="utf-8-sig")
texts = df["processed_final_review"].dropna().astype(str).tolist()

# 3) Build stopword set (NLTK + domain + manual noise)
base_stop    = set(stopwords.words("english"))
domain_stop  = {"hotel","room","stay","place","good","nice","like","one","would","could"}
general_stop = {"clean","thank","go","say","great","wonderful","bad"}
manual_stop  = {"river","mind"}
STOP = base_stop | domain_stop | general_stop | manual_stop

# 4) Preprocess + POS‑filter: keep only NOUN/PROPN tokens
def preprocess(doc):
    tokens = []
    for w in word_tokenize(doc.lower()):
        if w.isalpha() and w not in STOP:
            sp = nlp(w)[0]
            if sp.pos_ in {"NOUN","PROPN"}:
                tokens.append(w)
    return tokens

tok_texts = [preprocess(t) for t in texts]

# 5) Find optimal number of topics by coherence
dictionary = Dictionary(tok_texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(doc) for doc in tok_texts]

best_k, best_coh = 0, -1
for k in [5,6,7]:
    lda = LdaModel(corpus, id2word=dictionary, num_topics=k, passes=15, random_state=42)
    coh = CoherenceModel(model=lda, texts=tok_texts, dictionary=dictionary, coherence="c_v") \
          .get_coherence()
    print(f"k={k} → coherence={coh:.4f}")
    if coh > best_coh:
        best_k, best_coh = k, coh

print(f"\n# Selected num_topics = {best_k}, coherence={best_coh:.4f}\n")
lda = LdaModel(corpus, id2word=dictionary, num_topics=best_k, passes=15, random_state=42)

# 6) Collect all topic keywords (pruned)
keywords = set()
for t in range(best_k):
    for w,_ in lda.show_topic(t, topn=15):
        if w not in STOP:
            keywords.add(w)
keywords = list(keywords)

# 7) KMeans clustering of those keywords via spaCy vectors
embs, words = [], []
for w in keywords:
    tok = nlp(w)
    if tok.has_vector:
        embs.append(tok.vector)
        words.append(w)
X = np.vstack(embs)

n_clusters = 6
km = KMeans(n_clusters=n_clusters, random_state=42).fit(X)
centroids = km.cluster_centers_

clusters = {i: [] for i in range(n_clusters)}
for i,w in enumerate(words):
    lbl = km.labels_[i]
    sim = float(np.dot(embs[i], centroids[lbl]) /
                (np.linalg.norm(embs[i]) * np.linalg.norm(centroids[lbl])))
    clusters[lbl].append((w, sim))

# Keep top‑10 by similarity in each cluster
for lbl in clusters:
    clusters[lbl] = [w for w,_ in sorted(clusters[lbl], key=lambda x:-x[1])[:10]]

# 8) Seed‑Lexicon Expansion: top 12 similar NOUNs to each seed
seed_aspects = ["room","service","location","price","food"]
lexicons = {}
freq_tokens = pd.Series([w for doc in tok_texts for w in doc]).value_counts().index.tolist()[:1000]

for seed in seed_aspects:
    vec_s = nlp(seed).vector
    sims = []
    for w in freq_tokens:
        tok = nlp(w)[0]
        if tok.has_vector and tok.pos_ in {"NOUN","PROPN"}:
            s = float(np.dot(vec_s, tok.vector) /
                      (np.linalg.norm(vec_s)*np.linalg.norm(tok.vector)))
            if s >= 0.60:
                sims.append((w,s))
    lexicons[seed] = [w for w,_ in sorted(sims, key=lambda x:-x[1])[:12]]

# 9) Keyphrase extraction via noun‑chunk + KeyBERT
grammar = "NP: {<JJ>*<NN.*>+}"
chunker = RegexpParser(grammar)

def get_noun_phrases(text):
    nps = []
    for sent in sent_tokenize(text):
        tags = pos_tag(word_tokenize(sent.lower()))
        tree = chunker.parse(tags)
        for subtree in tree.subtrees():
            if subtree.label()=="NP":
                nps.append(" ".join(w for w,_ in subtree.leaves()))
    return nps

all_chunks = [" ".join(get_noun_phrases(doc)) for doc in texts]
keyphrases = kw_model.extract_keywords(
    all_chunks, top_n=20,
    keyphrase_ngram_range=(1,2),
    stop_words=list(STOP)
)

# 10) Print everything
print("=== Seed‑Lexicons ===")
for asp, kws in lexicons.items():
    print(f"{asp:10}: {kws}")

print("\n=== Discovered Aspect Clusters ===")
for i, kws in clusters.items():
    print(f"Aspect {i+1}: {kws}")

print("\n=== Suggested Keyphrases ===")
print(keyphrases)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


k=5 → coherence=0.4702
k=6 → coherence=0.4679
k=7 → coherence=0.4875

# Selected num_topics = 7, coherence=0.4875



LookupError: 
**********************************************************************
  Resource [93maveraged_perceptron_tagger_eng[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('averaged_perceptron_tagger_eng')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtaggers/averaged_perceptron_tagger_eng/[0m

  Searched in:
    - 'C:\\Users\\catsu/nltk_data'
    - 'C:\\Users\\catsu\\AppData\\Local\\Programs\\Python\\Python311\\nltk_data'
    - 'C:\\Users\\catsu\\AppData\\Local\\Programs\\Python\\Python311\\share\\nltk_data'
    - 'C:\\Users\\catsu\\AppData\\Local\\Programs\\Python\\Python311\\lib\\nltk_data'
    - 'C:\\Users\\catsu\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


In [ ]:
1. Beş “Seed” Aspect (Domain‑Temelli)

Aspect	Örnek Kelimeler
room	room, floor, bedroom, bathroom, kitchen, balcony, bed, apartment, desk, hall, laundry, sofa, basement, spacious
service	service, staff, customer, maintenance, internet, support, quality, cleanliness, reliable
location	location, proximity, area, destination, vicinity, distance, close, parking
price	price, cost, value, discount, affordable, expensive, pay, worth
food	food, meal, restaurant, breakfast, lunch, dinner, soup, pizza, bread, coffee, dessert


Aspect	Anahtar Kelimeler	Yorum
Beach & Water	beach, sea, water, pool	Sahil/deniz termal ve havuz odaklı
Accommodation	apartment, house, accommodation	Konaklama biriminin kendisi (oda/apart)
Facilities & Cleanliness	bathroom, shower, toilet, towel, door, cleanliness	Tesis içi donanım + temizlik kalitesi

Accommodation & Facilities
– Kelime listesi: bathroom, shower, apartment, hotel, location
– Açıklama: Konaklama biriminin fiziksel özellikleri ve olanakları (oda, banyo, duş, konum vb.)

Service & Social Experience
– Kelime listesi: staff, place, family, people, night, problem, river, price
– Açıklama: Personel/misafir etkileşimleri, çevresel deneyimler ve fiyat/şikayet gibi konular

Leisure & Meals
– Kelime listesi: breakfast, beach, holiday
– Açıklama: Yeme‐içme ve tatil‐dinlenme aktiviteleri


3. Öne Çıkan Keyphrases (2‑Tok Macro‑Öbekler)
KeyBERT + noun‑chunking ile çıkardığım en güçlü 1–2 kelimelik öbekler:

room size, breakfast service, beach view, kitchen facilities, customer service, pool area, bed comfort, location map, price value, food quality
Bunlar UI’da alternatif hızlı filtre seçenekleri olabilir.
                                                                    